> **5 minutes. One YAML file. Every row accounted for.**

# LakeLogic — Stop Your Pipeline From Silently Dropping Rows (and Leaking PII)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/00_quickstart.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/00_quickstart.ipynb)

Most data quality bugs aren't loud — they're silent. Bad rows get filtered, dropped, or quietly NULLed. PII lands in the warehouse unmasked. Nobody notices until finance reconciles three months later — or legal does.

This notebook shows how to catch every bad row, quarantine it, mask PII automatically, and prove nothing was lost — in 30 lines of YAML, zero custom Python.

In [14]:
# Install lakelogic
!pip install -q lakelogic[polars,duckdb]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip


### ⚙️ Pick an Engine

Same contract runs on **Polars**, **DuckDB**, or **Spark**. No code change — just flip the flag.

In [15]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "polars"  # 'polars' , 'duckdb', 'spark'

## The Problem

Raw `orders` land in your lake every hour. Some rows have:

- Bad emails (`user@@gmail`)
- Negative amounts (`-£42.00`)
- Statuses you've never seen (`'shippd'`, `'REFUNDED'`)
- Missing IDs
- **PII in plain text** that shouldn't reach the silver layer

Today, the bad rows either crash your DAG at 3am or — worse — get silently filtered out and you ship a dashboard that's 12% wrong. Meanwhile, full customer emails sit in your warehouse waiting for your next audit.

**You need to validate every row, quarantine the bad ones, mask the PII, and prove all three happened.** With zero custom Python.

## The Solution

One YAML contract. It defines the schema, the quality rules, and a derived column. Run it through `DataProcessor` and you get back two dataframes — `good` and `bad` — that always sum back to the source.

In [16]:
contract = s.write_contract(
    """
version: 1.0.0
dataset: orders

info:
  title: E-Commerce Orders
  owner: data-team@company.com
  target_layer: silver

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: customer_email
      type: string
      required: true
      pii: true
      masking: partial   # GDPR: mask in silver, keep originals in bronze
    - name: amount
      type: float
      required: true
    - name: currency
      type: string
    - name: status
      type: string
    - name: created_at
      type: string

transformations:
  - phase: "post"
    derive:
      field: "amount_gbp"
      sql: "CAST(CASE WHEN currency='USD' THEN amount*0.79 WHEN currency='EUR' THEN amount*0.86 ELSE amount END AS DECIMAL(10,2))"

quality:
  row_rules:
    - name: valid_email
      sql: "customer_email LIKE '%@%.%'"
    - name: positive_amount
      sql: "amount > 0"
    - name: valid_status
      sql: "status IN ('pending','shipped','delivered','returned')"
    - name: valid_currency
      sql: "currency IN ('GBP','USD','EUR')"
    - name: valid_order_id
      sql: "order_id > 0"

""",
    "00_quickstart_demo/orders_contract.yaml",
)

# Generate 1000 rows — 10% intentionally bad
source_df = ll.DataGenerator(contract).generate(rows=1000, invalid_ratio=0.10, output_format=ENGINE)

# Run the pipeline
proc = ll.DataProcessor(contract, engine=ENGINE)
good, bad = proc.run(source_df)
good, bad = s.to_polars(good), s.to_polars(bad)

2026-05-20 21:09:17.755 | INFO     | lakelogic.core.generator:generate:3422 - 📋 Generating data for: E-Commerce Orders
2026-05-20 21:09:17.755 | INFO     | lakelogic.core.generator:generate:3423 -    Records    : 900 valid + 100 invalid = 1,000 total
2026-05-20 21:09:17.757 | INFO     | lakelogic.core.generator:generate:3439 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-05-20 21:09:17.757 | INFO     | lakelogic.core.generator:generate:3454 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-05-20 21:09:17.827 | INFO     | lakelogic.core.generator:generate:3488 -    Row generation complete: 1,000 records built
2026-05-20 21:09:17.829 | INFO     | lakelogic.core.generator:generate:3510 -    Test cases : 232 across 7 categories
2026-05-20 21:09:17.830 | INFO     | lakelogic.core.generator:generate:3512 -      NOT_NULL_VIOLATION               92 injections
2026-05-20 21:09:17.830 | INFO     | lakelogic.core.generator:generate:3512 -      EMPTY_STR

## The Proof

Four things every data pipeline should give you for free — and almost none do:

1. **Reconciliation** — source rows == good + bad, always
2. **Quarantine** — every bad row preserved with the reason it failed
3. **PII masking** — applied automatically to `good` rows (raw values preserved in `bad` for investigation)
4. **Audit trail** — run ID, per-rule failure counts, timing

In [17]:
# Every row accounted for
s.assert_reconciliation(source_df, good, bad)

source=1000  good=879  bad=121
1000 == 879 + 121 -> True


In [18]:
# What was good
print("Valid rows (sample):")
display(good.head(5))

Valid rows (sample):


order_id,customer_email,amount,currency,status,created_at,_is_invalid,_test_case_types,amount_gbp
i64,str,f64,str,str,str,bool,str,"decimal[10,2]"
8710,"""q***@pigr.com""",189.7,"""GBP""","""pending""","""2026-04-08T01:24:13.783613""",false,null,189.70
9834,"""c***@mxnoa.com""",142.32,"""USD""","""delivered""","""2026-04-27T17:38:29.810002""",false,null,112.43
8205,"""o***@qhk.net""",49.03,"""EUR""","""pending""","""2026-03-03T13:30:05.800368""",false,null,42.17
8154,"""u***@xrfmrwp.com""",97.21,"""USD""","""delivered""","""2026-03-24T04:50:36.778058""",false,null,76.80
6556,"""s***@omuusmt.io""",338.7,"""USD""","""returned""","""2026-03-27T01:31:07.786200""",false,null,267.57


In [19]:
# What was caught
print("Quarantined rows (sample):")
display(bad.head(5))

Quarantined rows (sample):


order_id,customer_email,amount,currency,status,created_at,_is_invalid,_test_case_types,amount_gbp,_lakelogic_errors,_lakelogic_categories,quarantine_state,quarantine_reprocessed
i64,str,f64,str,str,str,bool,str,"decimal[10,2]",list[str],list[str],str,bool
null,"""wdnigsrmth@emkn.net""",33.1,null,"""INVALID_JNBY""","""2026-04-07T18:47:55.825772""",true,"""ACCEPTED_VALUE_VIOLATION,NOT_NULL_VIOLATION""",33.10,"[""Rule failed: order_id_required (""order_id"" IS NOT NULL)"", ""Rule failed: valid_status (status IN ('pending','shipped','delivered','returned'))"", … ""Rule failed: valid_order_id (order_id > 0)""]","[""completeness"", ""correctness"", … ""correctness""]","""active""",false
3098,"""ldelcqil@nbvdnc.net""",33.38,null,"""delivered""","""2026-02-27T06:35:01.788199""",false,null,33.38,"[""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","[""correctness""]","""active""",false
6695,""" """,null,"""EUR""","""pending""",null,true,"""EDGE_CASE_BUILTIN,NOT_NULL_VIOLATION""",null,"[""Rule failed: amount_required (""amount"" IS NOT NULL)"", ""Rule failed: valid_email (customer_email LIKE '%@%.%')"", ""Rule failed: positive_amount (amount > 0)""]","[""completeness"", ""correctness"", ""correctness""]","""active""",false
5420,"""xgfalvyln@lhtroru.io""",142.05,"""INVALID_XOFI""","""shipped""","""2026-05-15T23:45:45.826668""",true,"""ACCEPTED_VALUE_VIOLATION""",142.05,"[""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))""]","[""correctness""]","""active""",false
null,"""nqdlgcbwi@wkzh.io""",2.01,"""""","""pending""","""2026-05-16T14:49:07.826668""",true,"""EMPTY_STRING,NOT_NULL_VIOLATION""",2.01,"[""Rule failed: order_id_required (""order_id"" IS NOT NULL)"", ""Rule failed: valid_currency (currency IN ('GBP','USD','EUR'))"", ""Rule failed: valid_order_id (order_id > 0)""]","[""completeness"", ""correctness"", ""correctness""]","""active""",false


In [20]:
# Full audit trail
s.print_report(proc)

Run ID      : 670debbd-9a13-48c9-ae78-ae19a30c35db
Timestamp   : 2026-05-20T20:09:17+00:00
Source      : 1000
Good        : 879
Quarantined : 121

Rule failures:
  order_id_required: 20 rows
  valid_status: 42 rows
  valid_currency: 73 rows
  valid_order_id: 41 rows
  amount_required: 29 rows
  valid_email: 31 rows
  positive_amount: 37 rows
  customer_email_required: 6 rows


{'run_id': '670debbd-9a13-48c9-ae78-ae19a30c35db',
 'pipeline_run_id': None,
 'engine': 'polars',
 'contract': 'E-Commerce Orders',
 'contract_file_name': None,
 'contract_version': '1.0.0',
 'stage': 'default',
 'dataset': 'orders',
 'domain': None,
 'system': None,
 'environment': 'local',
 'data_layer': 'silver',
 'source_path': None,
 'source_files': [],
 'max_source_mtime': None,
 'timestamp': '2026-05-20T20:09:17+00:00',
 'counts': {'source': 1000,
  'total': 1000,
  'good': 879,
  'quarantined': 121,
  'quarantine_ratio': 0.121,
  'pre_transform_dropped': 0},
 'dataset_rules': [],
 'slos': {},
 'row_rule_failures': [{'name': 'order_id_required',
   'sql': '"order_id" IS NOT NULL',
   'message': 'Rule failed: order_id_required ("order_id" IS NOT NULL)',
   'count': 20},
  {'name': 'valid_status',
   'sql': "status IN ('pending','shipped','delivered','returned')",
   'message': "Rule failed: valid_status (status IN ('pending','shipped','delivered','returned'))",
   'count': 42},
 

## What You Just Did

In ~30 lines of YAML, you got:

- ✅ **Schema enforcement** — typed fields, required flags, PII tagging
- ✅ **5 quality rules** — declarative SQL, not nested `if`s
- ✅ **Automatic PII masking** — `customer_email` → `v***@example.com` in the output
- ✅ **A derived column** (`amount_gbp`) — computed inside the contract
- ✅ **100% reconciliation** — `source == good + bad`, mathematically guaranteed
- ✅ **A full audit trail** — run ID, per-rule failure counts, timing
- ✅ **Engine portability** — same contract on Polars, DuckDB, or Spark

The custom Python you would have written: **zero lines**.

The hours of GDPR remediation you just avoided: **a lot**.

---
## Go Deeper — Explore by Capability

Each notebook below is **self-contained** and maps to one pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities). Pick the one that matters to you most.

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

---

**Like what you saw?** ⭐ [Star us on GitHub](https://github.com/LakeLogic/LakeLogic) — it's how we know this matters.